# BERT Sentiment Model for Graduation Thesis
# 毕业论文：BERT 情感分类模型

This notebook fine-tunes a **BERT-based binary sentiment classifier** on the **same 60/10/30 split** used by the baseline TF-IDF + Logistic Regression model and the BiLSTM model.

本 notebook 在与你前面模型完全一致的 **60/10/30** 数据划分上，微调一个 **BERT 二分类情感模型**，并复用相同的 **Neutral 阈值策略**：

- `NEG_TH = 0.40`
- `POS_TH = 0.60`

Expected outputs / 预期输出：

- `artifacts/models/bert_model/`
- `artifacts/predictions/pred_test_bert_binary.csv`
- `artifacts/predictions/pred_test_bert_neutral_0.40_0.60.csv`

> **Important / 重要**
>
> This notebook uses the raw `text` column as BERT input.  
> 本 notebook 使用原始 `text` 作为 BERT 输入。


In [5]:
# If needed, install dependencies in your current environment.
# 如有需要，请先在当前环境安装依赖。
# !pip install -U transformers datasets accelerate scikit-learn pandas torch


## 1. Setup and imports
## 1. 环境配置与依赖导入

We import the required libraries for:
- loading data / 读取数据
- tokenization / 文本编码
- fine-tuning BERT / 微调 BERT
- evaluation / 模型评估
- exporting outputs / 导出结果


In [1]:
# =====================================
# Setup and imports
# 环境配置与依赖导入
# =====================================

import os

# Force Transformers to use PyTorch only
# 强制 Transformers 只走 PyTorch，不加载 TensorFlow / Flax
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)

## 2. Paths and experiment configuration
## 2. 路径与实验参数设置

All paths are aligned with your current project structure.

这里的路径完全与你当前的项目结构保持一致。


In [2]:
# ------------------------------
# Paths / 路径
# ------------------------------
BASE_DIR = Path(".")
ARTIFACT_DIR = BASE_DIR / "artifacts"
SPLIT_DIR = ARTIFACT_DIR / "splits"
MODEL_DIR = ARTIFACT_DIR / "models"
PRED_DIR = ARTIFACT_DIR / "predictions"
CONFIG_DIR = ARTIFACT_DIR / "config"

for p in [ARTIFACT_DIR, SPLIT_DIR, MODEL_DIR, PRED_DIR, CONFIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = SPLIT_DIR / "train_60.csv"
VAL_PATH   = SPLIT_DIR / "val_10.csv"
TEST_PATH  = SPLIT_DIR / "test_30.csv"

# ------------------------------
# Model / 模型
# ------------------------------
MODEL_NAME = "bert-base-uncased"

# ------------------------------
# Hyperparameters / 超参数
# ------------------------------
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
RANDOM_STATE = 42

# Neutral thresholds / 中性阈值
NEG_TH = 0.40
POS_TH = 0.60

# Reproducibility / 复现性
set_seed(RANDOM_STATE)
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
print("Train path:", TRAIN_PATH.resolve())
print("Val path:", VAL_PATH.resolve())
print("Test path:", TEST_PATH.resolve())


Device: cpu
Train path: /Users/victor/Desktop/Graduation_Thesis/Code/artifacts/splits/train_60.csv
Val path: /Users/victor/Desktop/Graduation_Thesis/Code/artifacts/splits/val_10.csv
Test path: /Users/victor/Desktop/Graduation_Thesis/Code/artifacts/splits/test_30.csv


## 3. Load split files
## 3. 读取划分数据

We reuse the exact same train / validation / test split generated earlier.

这里复用你已经固定好的 train / validation / test 划分，以保证 LR / LSTM / BERT 对比公平。


In [3]:
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nColumns in train_df:", list(train_df.columns))
train_df.head(3)


Train shape: (960000, 2)
Val shape: (160000, 2)
Test shape: (480000, 2)

Columns in train_df: ['text_clean', 'label']


,text_clean,label
0,thx user 100listeners!thk you all hi user user...,1
1,ergh miserable weather,0
2,user apple's in ears are slightly comfy for me...,1


## 4. Basic schema checks
## 4. 基本字段检查

We expect:
- `text` for raw text / 原始文本
- `label` for binary labels / 二分类标签

`text_clean` is optional here for BERT training, but will still be exported for later comparison and analysis.

这里 `text_clean` 对 BERT 训练不是必须的，但为了后续分析，我们仍会一并导出。


In [6]:
required_cols = ["text", "label"]
missing_cols = [c for c in required_cols if c not in train_df.columns]


if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

if "text_clean" not in train_df.columns:
    print("Warning: 'text_clean' not found. The notebook will fall back to raw 'text' when exporting.")


## 5. Build Hugging Face datasets
## 5. 构建 Hugging Face 数据集

BERT uses the raw `text` directly and relies on the pretrained tokenizer.

BERT 直接使用原始 `text`，并依赖预训练 tokenizer 做文本编码。


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def build_hf_dataset(df: pd.DataFrame) -> Dataset:
    text_clean_fallback = df["text_clean"] if "text_clean" in df.columns else df["text"]
    data = {
        "text": df["text"].astype(str).tolist(),
        "text_clean": text_clean_fallback.astype(str).tolist(),
        "label": df["label"].astype(int).tolist(),
    }
    return Dataset.from_dict(data)

train_ds_raw = build_hf_dataset(train_df)
val_ds_raw   = build_hf_dataset(val_df)
test_ds_raw  = build_hf_dataset(test_df)

train_ds_raw


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Dataset({
    features: ['text', 'text_clean', 'label'],
    num_rows: 960000
})

## 6. Tokenization
## 6. 文本编码

We tokenize the raw text with the BERT tokenizer.
We keep truncation on and let the data collator handle dynamic padding.

这里使用 BERT tokenizer 对原始文本编码，并采用动态 padding。


In [8]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=MAX_LEN,
    )

train_ds = train_ds_raw.map(tokenize_batch, batched=True)
val_ds   = val_ds_raw.map(tokenize_batch, batched=True)
test_ds  = test_ds_raw.map(tokenize_batch, batched=True)

train_ds = train_ds.rename_column("label", "labels")
val_ds   = val_ds.rename_column("label", "labels")
test_ds  = test_ds.rename_column("label", "labels")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_ds


Map:   0%|          | 0/960000 [00:00<?, ? examples/s]

Map:   0%|          | 0/160000 [00:00<?, ? examples/s]

Map:   0%|          | 0/480000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'text_clean', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 960000
})

## 7. Load pretrained BERT model
## 7. 加载预训练 BERT 模型

We fine-tune `bert-base-uncased` as a binary sentiment classifier with `num_labels=2`.

这里使用 `bert-base-uncased` 作为二分类情感分类器进行微调。


In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print(model.__class__.__name__)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification


## 8. Define evaluation metrics
## 8. 定义评估指标

We use the same core metrics as the previous models:
- accuracy
- precision
- recall
- F1

这里保持与 baseline 和 LSTM 一致的评估指标。


In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


## 9. Training arguments
## 9. 训练参数设置

The configuration is intentionally simple and stable:
- 2 epochs
- batch size = 16
- learning rate = 2e-5

这里保持参数设置简洁稳定，便于顺利完成实验和论文复现。


In [12]:
OUTPUT_DIR = ARTIFACT_DIR / "bert_training_outputs"

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    report_to="none",
    seed=RANDOM_STATE,
)


## 10. Build Trainer
## 10. 构建 Trainer

The Hugging Face `Trainer` API handles training, validation, and prediction in a clean way.

Hugging Face 的 `Trainer` 可以统一处理训练、验证和预测流程。


In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)


/var/folders/y2/7skrqxhn4755hd3ljqchjx140000gn/T/ipykernel_21708/1917998872.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


## 11. Fine-tune BERT
## 11. 微调 BERT

This is the main training step.

这是核心训练过程。


In [15]:
train_result = trainer.train()
train_result


  0%|          | 0/120000 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 12. Evaluate on the test set
## 12. 在测试集上评估

We obtain:
- logits
- positive-class probability `P(pos)`
- binary predictions

这里得到：
- logits
- 正类概率 `P(pos)`
- 二分类预测结果


In [ ]:
test_output = trainer.predict(test_ds)

logits = test_output.predictions
y_true = np.array(test_ds["labels"])

probs_tensor = torch.softmax(torch.tensor(logits), dim=1)
proba_pos = probs_tensor[:, 1].numpy()

pred_binary = np.argmax(logits, axis=1)

print("Classification report / 分类报告:")
print(classification_report(y_true, pred_binary, digits=4))

print("\nConfusion matrix / 混淆矩阵:")
print(confusion_matrix(y_true, pred_binary))


## 13. Neutral post-processing strategy
## 13. 中性标签后处理策略

Like the earlier notebooks, the model is trained as binary classification,
but an uncertain probability region is mapped to `NEUTRAL`.

与前面的 notebook 一样，模型训练仍为二分类，
但通过概率阈值将不确定区间映射为 `NEUTRAL`。


In [ ]:
def proba_to_3class(probs, neg_th=0.40, pos_th=0.60):
    labels = []
    for p in probs:
        if p <= neg_th:
            labels.append("NEGATIVE")
        elif p >= pos_th:
            labels.append("POSITIVE")
        else:
            labels.append("NEUTRAL")
    return labels

pred_3class = proba_to_3class(proba_pos, neg_th=NEG_TH, pos_th=POS_TH)

neutral_rate = np.mean(np.array(pred_3class) == "NEUTRAL")
print(f"Neutral rate on test set / 测试集中的中性比例: {neutral_rate:.4f}")


## 14. Export prediction files
## 14. 导出预测结果文件

Output files:
- `artifacts/predictions/pred_test_bert_binary.csv`
- `artifacts/predictions/pred_test_bert_neutral_0.40_0.60.csv`

这里的输出路径已经与你目前的工程结构完全一致。


In [ ]:
pred_binary_df = pd.DataFrame({
    "text": test_df["text"].astype(str).tolist(),
    "text_clean": (
        test_df["text_clean"] if "text_clean" in test_df.columns else test_df["text"]
    ).astype(str).tolist(),
    "y_true": y_true.astype(int),
    "proba_pos": proba_pos.astype(float),
    "pred_binary": pred_binary.astype(int),
})

pred_neutral_df = pred_binary_df.copy()
pred_neutral_df["pred_3class"] = pred_3class

binary_path = PRED_DIR / "pred_test_bert_binary.csv"
neutral_path = PRED_DIR / f"pred_test_bert_neutral_{NEG_TH:.2f}_{POS_TH:.2f}.csv"

pred_binary_df.to_csv(binary_path, index=False, encoding="utf-8")
pred_neutral_df.to_csv(neutral_path, index=False, encoding="utf-8")

print("Saved binary preds / 已保存二分类预测:", binary_path.resolve())
print("Saved neutral preds / 已保存中性策略预测:", neutral_path.resolve())


## 15. Save BERT model and tokenizer
## 15. 保存 BERT 模型与 tokenizer

We save both the model and tokenizer to:

`artifacts/models/bert_model/`

模型和 tokenizer 都会被保存到该目录中，便于后续复现和加载。


In [ ]:
BERT_SAVE_DIR = MODEL_DIR / "bert_model"
BERT_SAVE_DIR.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(BERT_SAVE_DIR)
tokenizer.save_pretrained(BERT_SAVE_DIR)

print("Saved BERT model to / 已保存 BERT 模型至:", BERT_SAVE_DIR.resolve())


## 16. Optional quick demo
## 16. 可选：快速推理演示

You can try a few custom sentences below.
你可以在下面测试几条自定义句子。


In [ ]:
demo_texts = [
    "I am so happy today!",
    "This product is terrible and disappointing.",
    "It is okay, I guess."
]

demo_enc = tokenizer(
    demo_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LEN,
    return_tensors="pt"
)

with torch.no_grad():
    demo_logits = trainer.model(**demo_enc).logits
    demo_probs = torch.softmax(demo_logits, dim=1)[:, 1].cpu().numpy()
    demo_binary = np.argmax(demo_logits.cpu().numpy(), axis=1)
    demo_3class = proba_to_3class(demo_probs, NEG_TH, POS_TH)

list(zip(demo_texts, demo_probs, demo_binary, demo_3class))


# Notebook summary
# Notebook 组成说明

This notebook contains five main components:

本 notebook 主要由五个部分组成：

### 1. Data loading / 数据读取
- Reuses the fixed 60/10/30 split
- 复用你已经固定好的 60/10/30 数据划分

### 2. Tokenization / 文本编码
- Uses the BERT tokenizer on raw `text`
- 使用 BERT tokenizer 对原始 `text` 编码

### 3. Fine-tuning / 模型微调
- Fine-tunes `bert-base-uncased` for binary sentiment classification
- 对 `bert-base-uncased` 进行二分类微调

### 4. Neutral strategy / 中性后处理
- Converts `P(pos)` into NEGATIVE / NEUTRAL / POSITIVE
- 将 `P(pos)` 进一步映射成 NEGATIVE / NEUTRAL / POSITIVE

### 5. Export artifacts / 导出结果
- Saves prediction CSV files to `artifacts/predictions/`
- Saves the model to `artifacts/models/bert_model/`
- 将预测结果保存到 `artifacts/predictions/`
- 将模型保存到 `artifacts/models/bert_model/`

This keeps the BERT experiment fully aligned with your previous LR and LSTM notebooks.

这保证了 BERT 与之前的 LR / LSTM 实验在路径、输出结构和比较方式上完全一致。
